# SIKAFE — M2 en Kaggle: entrenar y evaluar YOLO con fotos reales de café (Croppie)

Versión para Kaggle del notebook de Colab (mismo código del repositorio, misma semilla, mismos resultados esperables).

**Configuración (panel derecho → *Session options* / *Settings*):**
1. **Accelerator: GPU T4 x2** (o P100).
2. **Internet: On** (necesario para descargar el código y el dataset).
   Ambas opciones exigen tener el **teléfono verificado** en la cuenta de Kaggle.

**Cómo ejecutarlo sin dejar la pestaña abierta:** botón **Save Version → Save & Run All (Commit)**.
Corre en segundo plano (máximo ~12 h por sesión). Al terminar, en la versión guardada verá la salida de cada celda
y en la pestaña **Output** los archivos de la corrida (pesos, métricas, figuras).

La evaluación FINAL en test **no** se ejecuta en esta pasada (la decidimos después de revisar validación).

In [ ]:
# 1. Comprobar la GPU
!nvidia-smi

In [ ]:
# 2. Carpetas: datos temporales en /tmp; resultados en /kaggle/working (se guardan en Output)
SALIDA = '/kaggle/working/sikafe'
RAW = '/tmp/croppie_raw'
DATOS = '/tmp/sikafe_yolo'
!mkdir -p {SALIDA}/corridas {SALIDA}/datos

In [ ]:
# 3. Código del proyecto (rama subida a GitHub)
%cd /tmp  # salir de la carpeta antes de borrarla (si no, git clone falla al re-ejecutar)
RAMA = 'fase-2-3-croppie-yolo'
!rm -rf /tmp/rendimientocosecha
!git clone -q -b {RAMA} https://github.com/Darwin-md11/rendimientocosecha /tmp/rendimientocosecha
%cd /tmp/rendimientocosecha
!git log --oneline -1

In [ ]:
# 4. Dependencias del laboratorio (Kaggle ya trae PyTorch con CUDA)
!pip install -q -r requirements-ml.txt
import torch
import ultralytics

print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| CUDA', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No hay GPU: active Accelerator = GPU en Settings'

In [ ]:
# 5. Descargar Croppie desde Hugging Face (~931 MB)
import os
import zipfile

from huggingface_hub import hf_hub_download

for archivo in ['data/train.zip', 'data/val.zip']:
    ruta = hf_hub_download('rgautroncgiar/croppie_coffee_ug', archivo, repo_type='dataset')
    destino = os.path.join(RAW, os.path.basename(archivo)[:-4])
    zipfile.ZipFile(ruta).extractall(destino)
    print('extraído', archivo, '->', destino)
!find {RAW} -maxdepth 3 -type d | head -20

### 📋 6. Inspeccionar nombres y agrupamiento

In [ ]:
!python -m ml.datos.preparar_yolo --origen {RAW} --inspeccionar

### 📋 7. Preparar el dataset
Si el agrupamiento no es plausible, esta celda termina con `ERROR` y el entrenamiento no se ejecuta: pegue la salida de las celdas 6 y 7 en el chat.

In [ ]:
import subprocess
import sys

r = subprocess.run([sys.executable, '-m', 'ml.datos.preparar_yolo', '--origen', RAW, '--destino', DATOS, '--dataset', 'croppie', '--semilla', '0'],
                   capture_output=True, text=True)
print(r.stdout[-6000:], r.stderr[-3000:])
assert r.returncode == 0, 'La preparación falló: copie la salida de las celdas 6 y 7 en el chat'
!cp {DATOS}/eda.md {DATOS}/eda_clases.png {DATOS}/estadisticas.json {DATOS}/manifiesto.json {SALIDA}/datos/

### 8. Entrenar YOLO11n a 640 px

In [ ]:
!python -m ml.m2_yolo.entrenar --config ml/experiments/configs/m2_yolo11n_640.yaml --datos {DATOS}/data.yaml --salida {SALIDA}/corridas

### 📋 9. Evaluar en validación

In [ ]:
import glob

CORRIDA = sorted(glob.glob(f'{SALIDA}/corridas/m2_yolo11n_640_*'))[-1]
print('Corrida:', CORRIDA)
!python -m ml.m2_yolo.evaluar --corrida {CORRIDA} --split val

### 📋 10. Contrato del detector con el modelo real

In [ ]:
import glob
import sys

from PIL import Image

sys.path.insert(0, 'tests')
from contrato_detector import verificar_contrato  # noqa: E402

from src.infrastructure.adapters.detectors.yolo_detector_adapter import YoloFruitDetectorAdapter  # noqa: E402

det = YoloFruitDetectorAdapter.desde_archivo(f'{CORRIDA}/m2_yolo.pt')
img = Image.open(sorted(glob.glob(f'{DATOS}/images/val/*'))[0]).convert('RGB')
verificar_contrato(det, img)
dets = det.detectar_frutos(img, 0.25)
print('Contrato OK |', len(dets), 'cerezas en', img.size, '|', det.info())

### 11. Evaluación FINAL en test (una sola vez) — desactivada
Cambie a `True` **solo** cuando el equipo haya decidido que este es el modelo (después de revisar la celda 9).

In [ ]:
EJECUTAR_TEST_FINAL = False
if EJECUTAR_TEST_FINAL:
    !python -m ml.m2_yolo.evaluar --corrida {CORRIDA} --split test --final
else:
    print('Test final NO ejecutado (protocolo: una sola vez, cuando se decida el modelo).')

In [ ]:
# 📋 12. Todo lo que hay que pegar en el chat
!cat {CORRIDA}/entorno.json
!cat {CORRIDA}/metricas_val_ultralytics.json
!cat {CORRIDA}/resumen_val.md
!cat {CORRIDA}/m2_yolo.manifest.json
!ls -la {CORRIDA}

### 13. Llevar el modelo a la app
En la versión guardada → pestaña **Output** → carpeta `sikafe/corridas/<id>/` → descargue **`m2_yolo.pt`** y **`m2_yolo.manifest.json`**
y cópielos a `models/` del repositorio en su computador.